# nano Beatrix v1.1 — before and after the frames

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_1_frames.ipynb)

The four nano v1.1 crafts trained on Beatrix 2s's diet, which holds none of the nano bed's framed rows: their frame reads show what plain text alone teaches about finding a record in context. This notebook continues each finished v1.1 craft for one more phase **with** the frames, beside the same continuation **without** them, so the before and the after are read on the same weights.

| craft (two a body) | starts from | its phase (1.0 billion bytes, 3,815 steps) |
|---|---|---|
| `nano-<body>-v1.1-fr` | the v1.1 craft's final state (step 61,422) | three rows in four framed, the rest plain fineweb-edu text |
| `nano-<body>-v1.1-ct` | the same state | every row plain (the control) |

for the four bodies `nano-pollux-dual-canon`, `nano-pollux-canon`, `nano-pollux-dual` and `nano-castor`.

**The start is exact:** each craft loads its v1.1 craft's last boundary state (the fp32 weights and both optimizers' states, `resume/boundary_anneal_mix_step61422.pt`) and continues at step 61,422 with the same recipe (Muon 0.02 on the trunk's matrices, Adam 0.0003 on the rest, the readout on AdamW with decay 0.1, the flat rate, clip 1.0, 262,144 bytes a step). The v1.1 folders are only read.

**The frames** are the nano bed's framed rows with the picks its re-run confirmed: one to three frames a row (lookup frames of up to eight `key: value` records; thinking frames of filler that nest a smaller lookup frame six times in ten), the sliced frame form (frames above 32 bytes built as 32-byte slices), frame sizes from 32 to 1,024 bytes, a key defined twice in three rows of ten (the later definition is the answer), the caption register, then the question and the value. A framed row is exactly one 4,096-byte block, so no frame is ever cut.

**The control reads the same text:** both arms draw every row from the same position of the same corpus (1.0 billion bytes of fineweb-edu, built in C2), so a plain row of the frames arm is the control's row byte for byte, and a framed row is built from the text at that position. The two arms differ only by the frames.

**The reads:** each craft's first boundary row is its v1.1 craft's end, read by the same gauges (the before); then the nano bed's gauges at steps 62,000 and 64,000 and at the end (the copy, the nested and redefined-key questions, the keyed record by frame size, the planted record out to 3,584 bytes, held-out bits), and the full eval on the fineweb-edu holdout every 2,000 steps. C4 tabulates before, after with frames, after the control, and the difference.

**Length:** about 1.7 hours for the four frames arms and the same for the four controls on an RTX PRO 6000 (the v1.1 benchmark: 0.30 to 0.49 s a step), plus the corpus build in C2 and about ten to fifteen minutes of before-reads at the start (an eval and a gauge for each of the eight crafts).

**Needs:** every v1.1 craft finished (C2 checks each final state is in the repo). **Token:** a Hugging Face token with write access to [AbstractPhil/alephllm-nano-castor-pollux](https://huggingface.co/AbstractPhil/alephllm-nano-castor-pollux) in this notebook's secrets as `HF_TOKEN`: the start states are read from it and the continuation folders ship to it.

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.35"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import nano_v1 as V1
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 35), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
assert HF_TOKEN, 'HF_TOKEN missing: the start states are read from the repo and the continuations ship to it'
os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK — every setting from the record, with its source beside it ============
OUT = '/content/nano_v1_1_frames'                     # the session folder: the corpus, the runs, the bed's gauge folder
REPO = V1.REPO                                        # AbstractPhil/alephllm-nano-castor-pollux: the v1.1 folders read, the new ones written
NAMES = list(V1.CRAFTS_CONT)                          # each body's frames arm then its control, so a stopped session leaves whole pairs
MB, GA = 16, 4                                        # the 2s shape, the fastest on the v1.1 benchmark (262,144 bytes a step)
GAUGE_EVERY = 2000                                    # the bed's gauges at 62,000 and 64,000 (every full eval) and at the end
CORPUS_BYTES = 1.0e9                                  # the frames corpus (fineweb-edu, the held-out head skipped), built once in C2
MAX_HOURS = None                                      # this session's budget in hours (None: until every craft ends or the cell is stopped)
print('decision block:', len(NAMES), 'crafts:', ', '.join(NAMES), '| the phase', V1.CONT_TOKENS / 1e9, 'B bytes |', MB, 'x', GA,
      '| gauges every', GAUGE_EVERY, 'steps | corpus', CORPUS_BYTES / 1e9, 'GB | ships to', REPO)

In [ ]:
# C2 (P) — PREFLIGHT: every v1.1 craft's final state in the repo, the held-out head, the frames corpus (built once; about a minute or two
# a gigabyte from the stream), the rows' composition pace against the card's, the push probe. Nothing trains here.
import json, time
from huggingface_hub import HfApi
from geolip.nano_beatrix.bed_corpus import write_heldout, build as build_corpus
from geolip.nano_beatrix import frames_data as FD
api = HfApi(token=HF_TOKEN)
files = set(api.list_repo_files(REPO))
missing = []
for n in NAMES:
    spec = V1.cont_init(n, MB * GA * V1.CTX)
    path = f"{spec['prefix']}/{spec['payload']}"
    if path not in files:
        missing.append(path)
assert not missing, f'not every v1.1 craft has finished: missing {sorted(set(missing))}'
print('start states: every v1.1 craft has its final state at step', spec['step'])
held = write_heldout(f'{OUT}/bed')
folder = f'{OUT}/frames_corpus'
if not os.path.exists(f'{folder}/bed_corpus.bin'):
    build_corpus(folder, int(CORPUS_BYTES))
packed = FD.load_corpus(folder)
n, t0 = 0, time.time()
while time.time() - t0 < 10:
    FD.row_bytes(packed, 1337, n, FD.P_FRAMED)
    n += 1
rate = n / (time.time() - t0)
need = 879_000 / FD.ROW_BYTES                         # the fastest craft's rows a second (Castor, 879k bytes a second)
print(f'frames corpus {len(packed) / 1e9:.2f} GB; the frames rows compose at {rate:,.0f} a second against {need:.0f} needed'
      + ('' if rate > 1.5 * need else ' — TOO SLOW: the card will wait on the data'))

PUSH_OK = False
path = 'preflight/push_probe_v1_1_frames.json'        # its own probe file
body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                   'compose_rows_s': rate, 'crafts': NAMES}).encode()
api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano v1.1 frames: push probe')
PUSH_OK = path in set(api.list_repo_files(REPO))
print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)

In [ ]:
# C3 (T) — THE CONTINUATIONS: each craft starts from its v1.1 craft's final state (its first row: the before, read by the gauges), then
# runs its phase; the trainer's lines and the bed's gauge lines ([v1]) print as they come. Rerun the cell to continue (a new session
# too). The cell's stop button saves the running craft's state before it stops.
assert PUSH_OK, 'the push probe did not read back: nothing would ship'
HIST = V1.run(OUT, HF_TOKEN, NAMES, micro_batch=MB, grad_accum=GA, repo=REPO, gauge_every=GAUGE_EVERY, max_hours=MAX_HOURS,
              frames_corpus=f'{OUT}/frames_corpus')
from IPython.display import Markdown, display
display(Markdown(V1.cont_table(HIST)))

In [ ]:
# C4 — BEFORE AND AFTER: the boundary rows of every continuation from the repo, the before-and-after table, and the curves. Run any
# time; it trains nothing.
import json
from IPython.display import Markdown, display
from huggingface_hub import snapshot_download
hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub',
                        allow_patterns=[f'{n}/runs/*' for n in NAMES] + [f'{n}/v1_boundaries.json' for n in NAMES])
hist = {}
for n in NAMES:
    p = f'{hub}/{n}/v1_boundaries.json'
    hist[n] = json.load(open(p, encoding='utf-8')) if os.path.exists(p) else []
display(Markdown(V1.cont_table(hist)))
display(Markdown(V1.table(hist)))
%load_ext tensorboard
%tensorboard --logdir {hub}

### Changing the run

Edit the decision block in C1, rerun C1 (the install is cached), then C2 and C3. A craft already in the repo resumes; it never restarts.

| to | the edit in C1 |
|---|---|
| one body first | `NAMES = ['nano-castor-v1.1-fr', 'nano-castor-v1.1-ct']` (the others wait) |
| a session budget | `MAX_HOURS = 2` |
| two cards | the first card `NAMES = NAMES[:4]`, the second `NAMES = NAMES[4:]` (two bodies each, every pair whole) |

The session folder is the Colab disk: the repo holds the records.